## Getting started

Run cells from top to bottom in Google Colab. When prompted, enter a descriptive SEC User-Agent containing your name or organization and contact email. The value is entered at runtime and is **not saved in this notebook**. SEC requests require a valid identifying User-Agent.

In [ ]:
# SEC EDGAR identification (entered only at runtime; never hardcode credentials or contact details)
from getpass import getpass

SEC_USER_AGENT = getpass("SEC User-Agent (name/organization and contact email): ").strip()
if not SEC_USER_AGENT or "@" not in SEC_USER_AGENT:
    raise ValueError("Provide a valid SEC User-Agent with contact email before running SEC API requests.")

In [ ]:

import requests
import pandas as pd

# Lockheed Martin's SEC Central Index Key (CIK)
cik = "0000936468"

url = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    f"CIK{cik}.json"
)

# Replace with your own identifying contact information.
headers = {
    "User-Agent": SEC_USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)
response.raise_for_status()

data = response.json()

print("Company:", data["entityName"])
print("Available taxonomies:", list(data["facts"].keys()))

gaap = data["facts"]["us-gaap"]

print("Number of US-GAAP concepts:", len(gaap))
print("Sample concepts:", list(gaap.keys())[:15])


## SEC extraction and Lockheed Martin exploratory analysis

Inspect SEC Company Facts concepts, build the initial historical financial dataset, and explore operating margins and segment results.

In [ ]:

revenue_tags = [
    tag for tag in gaap
    if "Revenue" in tag or "SalesRevenue" in tag
]

for tag in revenue_tags:
    print(tag)


In [ ]:

import pandas as pd

candidate_tags = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "Revenues",
    "SalesRevenueNet"
]

records = []

for tag in candidate_tags:
    if tag not in gaap:
        continue

    concept = gaap[tag]
    observations = concept["units"].get("USD", [])

    for obs in observations:
        if obs.get("form") != "10-K":
            continue

        if obs.get("fp") != "FY":
            continue

        if "start" not in obs or "end" not in obs:
            continue

        start = pd.to_datetime(obs["start"])
        end = pd.to_datetime(obs["end"])

        duration_days = (end - start).days

        # Keep approximately full-year reporting periods
        if not 350 <= duration_days <= 380:
            continue

        records.append({
            "tag": tag,
            "fiscal_year": obs.get("fy"),
            "period_start": obs["start"],
            "period_end": obs["end"],
            "revenue_billions": obs["val"] / 1e9,
            "filed": obs.get("filed"),
            "accession": obs.get("accn"),
            "frame": obs.get("frame")
        })

revenue_df = pd.DataFrame(records)

revenue_df = revenue_df[
    revenue_df["period_end"].str[:4].astype(int).between(2021, 2025)
]

revenue_df = revenue_df.sort_values(
    ["period_end", "tag", "filed"]
)

display(revenue_df)


In [ ]:

summary = (
    revenue_df
    .groupby(["period_end", "tag"], as_index=False)
    .agg(
        observations=("revenue_billions", "size"),
        min_revenue=("revenue_billions", "min"),
        max_revenue=("revenue_billions", "max"),
        latest_filing=("filed", "max")
    )
    .sort_values(["period_end", "tag"])
)

display(summary.round(3))


In [ ]:

# Keep the selected revenue concept
annual_revenue = revenue_df[
    revenue_df["tag"] == "Revenues"
].copy()

# Sort by reporting period and filing date
annual_revenue = annual_revenue.sort_values(
    ["period_end", "filed"]
)

# Keep the most recently filed observation
# for each reporting period
annual_revenue = annual_revenue.drop_duplicates(
    subset=["period_end"],
    keep="last"
)

# Use the actual period end to identify fiscal year
annual_revenue["year"] = pd.to_datetime(
    annual_revenue["period_end"]
).dt.year

# Calculate year-over-year growth
annual_revenue["revenue_growth_pct"] = (
    annual_revenue["revenue_billions"]
    .pct_change() * 100
)

# Create our clean reporting table
annual_revenue = annual_revenue[
    [
        "year",
        "revenue_billions",
        "revenue_growth_pct",
        "period_start",
        "period_end",
        "filed",
        "accession"
    ]
].reset_index(drop=True)

display(annual_revenue.round(3))


In [ ]:

starting_revenue = annual_revenue.iloc[0]["revenue_billions"]
ending_revenue = annual_revenue.iloc[-1]["revenue_billions"]

starting_year = int(annual_revenue.iloc[0]["year"])
ending_year = int(annual_revenue.iloc[-1]["year"])

years = ending_year - starting_year

total_growth = (
    (ending_revenue / starting_revenue) - 1
) * 100

cagr = (
    (ending_revenue / starting_revenue) ** (1 / years) - 1
) * 100

average_revenue = annual_revenue[
    "revenue_billions"
].mean()

print(f"Total Revenue Growth: {total_growth:.2f}%")
print(f"Revenue CAGR: {cagr:.2f}%")
print(f"Average Annual Revenue: ${average_revenue:.2f}B")


In [ ]:

search_terms = [
    "OperatingIncome",
    "NetIncome",
    "NetCashProvidedByUsedInOperatingActivities",
    "PaymentsToAcquirePropertyPlantAndEquipment",
    "Assets",
    "Liabilities"
]

for term in search_terms:
    print(f"\n--- {term} ---")

    matches = [
        tag for tag in gaap
        if term.lower() in tag.lower()
    ]

    for tag in matches[:15]:
        print(tag)


In [ ]:

import pandas as pd

financial_tags = {
    "revenue": "Revenues",
    "operating_income": "OperatingIncomeLoss",
    "net_income": "NetIncomeLoss",
    "operating_cash_flow": "NetCashProvidedByUsedInOperatingActivities",
    "capital_expenditures": "PaymentsToAcquirePropertyPlantAndEquipment",
    "total_assets": "Assets",
    "total_liabilities": "Liabilities"
}

# Balance sheet concepts are point-in-time values.
instant_metrics = {
    "total_assets",
    "total_liabilities"
}

def extract_annual_metric(gaap, metric_name, tag):
    if tag not in gaap:
        print(f"Missing SEC concept: {tag}")
        return pd.DataFrame()

    observations = gaap[tag]["units"].get("USD", [])
    records = []

    for obs in observations:
        if obs.get("form") != "10-K":
            continue

        period_end = obs.get("end")

        if not period_end:
            continue

        year = int(period_end[:4])

        if not 2021 <= year <= 2025:
            continue

        if metric_name not in instant_metrics:
            # Duration-based financial metrics
            if "start" not in obs:
                continue

            duration = (
                pd.to_datetime(obs["end"])
                - pd.to_datetime(obs["start"])
            ).days

            if not 350 <= duration <= 380:
                continue

        else:
            # Balance sheet values must be instantaneous
            if "start" in obs:
                continue

        records.append({
            "year": year,
            "metric": metric_name,
            "value": obs["val"],
            "filed": obs.get("filed"),
            "accession": obs.get("accn"),
            "period_end": period_end
        })

    if not records:
        print(f"No qualifying annual data: {metric_name}")
        return pd.DataFrame()

    df = pd.DataFrame(records)

    # Use the latest available filing for each year.
    df = df.sort_values(["year", "filed"])

    df = df.drop_duplicates(
        subset=["year"],
        keep="last"
    )

    return df


financial_records = []

for metric, tag in financial_tags.items():
    result = extract_annual_metric(
        gaap,
        metric,
        tag
    )

    if not result.empty:
        financial_records.append(result)

financial_long = pd.concat(
    financial_records,
    ignore_index=True
)

display(financial_long.head(15))


In [ ]:

financial_wide = financial_long.pivot(
    index="year",
    columns="metric",
    values="value"
).reset_index()

financial_wide.columns.name = None

# Convert dollar values to billions
for col in financial_wide.columns:
    if col != "year":
        financial_wide[col] = financial_wide[col] / 1e9

financial_wide = financial_wide.sort_values(
    "year"
).reset_index(drop=True)

display(financial_wide.round(3))


In [ ]:

# Cell 10 — Financial ratio calculations
# Version 2: handles missing capital expenditures

financial_analysis = financial_wide.copy()

# Revenue growth
financial_analysis["revenue_growth_pct"] = (
    financial_analysis["revenue"].pct_change() * 100
)

# Operating profit margin
financial_analysis["operating_margin_pct"] = (
    financial_analysis["operating_income"]
    / financial_analysis["revenue"] * 100
)

# Net profit margin
financial_analysis["net_profit_margin_pct"] = (
    financial_analysis["net_income"]
    / financial_analysis["revenue"] * 100
)

# Free cash flow requires capital expenditures
if "capital_expenditures" in financial_analysis.columns:
    financial_analysis["free_cash_flow"] = (
        financial_analysis["operating_cash_flow"]
        - financial_analysis["capital_expenditures"]
    )

    financial_analysis["fcf_margin_pct"] = (
        financial_analysis["free_cash_flow"]
        / financial_analysis["revenue"] * 100
    )
else:
    print(
        "Capital expenditures not yet available. "
        "Skipping free cash flow calculations."
    )

display(financial_analysis.round(3))


In [ ]:

# Cell 11 — Data quality checks

expected_metrics = [
    "revenue",
    "operating_income",
    "net_income",
    "operating_cash_flow",
    "capital_expenditures",
    "total_assets",
    "total_liabilities"
]

print("=== Missing Financial Metrics ===")

for metric in expected_metrics:
    if metric not in financial_wide.columns:
        print(f"MISSING COLUMN: {metric}")
    elif financial_wide[metric].isna().any():
        print(f"MISSING VALUES: {metric}")
    else:
        print(f"OK: {metric}")

print("\n=== Fiscal Years ===")
print(financial_wide["year"].tolist())

print("\n=== Missing Values ===")
print(financial_wide.isna().sum())

print("\n=== Financial Checks ===")

for _, row in financial_wide.iterrows():
    year = int(row["year"])

    if row["revenue"] <= 0:
        print(f"WARNING: Nonpositive revenue in {year}")

    if row["total_assets"] <= 0:
        print(f"WARNING: Nonpositive assets in {year}")

print("\nPreliminary data quality checks complete.")


In [ ]:

# Cell 12 — Investigate capital expenditure data

capex_tag = "PaymentsToAcquirePropertyPlantAndEquipment"

print("Tag exists:", capex_tag in gaap)

if capex_tag in gaap:
    concept = gaap[capex_tag]

    print("Available units:", list(concept["units"].keys()))

    observations = concept["units"].get("USD", [])

    print("Total USD observations:", len(observations))

    capex_df = pd.DataFrame(observations)

    if not capex_df.empty:
        columns = [
            "start", "end", "val", "form",
            "fy", "fp", "filed", "frame"
        ]

        available_columns = [
            col for col in columns
            if col in capex_df.columns
        ]

        display(
            capex_df[available_columns].tail(20)
        )

# Find other possible capital expenditure concepts
print("\n=== Alternative CapEx Tags ===")

alternatives = [
    tag for tag in gaap
    if any(term in tag.lower() for term in [
        "propertyplantandequipment",
        "capitalexpenditure",
        "paymentstoacquireproductive"
    ])
]

for tag in alternatives:
    print(tag)


In [ ]:

# Cell 13 — Investigate alternative CapEx tag

alternative_tag = "PaymentsToAcquireProductiveAssets"

print("Tag exists:", alternative_tag in gaap)

if alternative_tag in gaap:
    concept = gaap[alternative_tag]

    print("Description:")
    print(concept.get("description", "No description"))

    observations = concept["units"].get("USD", [])

    print("\nTotal observations:", len(observations))

    records = []

    for obs in observations:
        if obs.get("form") != "10-K":
            continue

        if "start" not in obs or "end" not in obs:
            continue

        year = int(obs["end"][:4])

        if not 2021 <= year <= 2025:
            continue

        duration = (
            pd.to_datetime(obs["end"])
            - pd.to_datetime(obs["start"])
        ).days

        if not 350 <= duration <= 380:
            continue

        records.append({
            "year": year,
            "start": obs["start"],
            "end": obs["end"],
            "capex_billions": obs["val"] / 1e9,
            "filed": obs.get("filed"),
            "accession": obs.get("accn")
        })

    alternative_capex_df = pd.DataFrame(records)

    if not alternative_capex_df.empty:
        alternative_capex_df = (
            alternative_capex_df
            .sort_values(["year", "filed"])
        )

        display(alternative_capex_df)

    else:
        print("No qualifying annual observations.")


In [ ]:

# Cell 14 — Compare CapEx observations

if not alternative_capex_df.empty:

    capex_summary = (
        alternative_capex_df
        .groupby("year", as_index=False)
        .agg(
            observations=("capex_billions", "size"),
            min_capex=("capex_billions", "min"),
            max_capex=("capex_billions", "max"),
            latest_filing=("filed", "max")
        )
    )

    capex_summary["values_agree"] = (
        (capex_summary["max_capex"]
         - capex_summary["min_capex"]).abs() < 0.000001
    )

    display(capex_summary.round(3))

else:
    print("No data available to compare.")


In [ ]:

# Cell 15 — Add capital expenditures to financial_wide

# Select the most recent filing for each year
capex_clean = (
    alternative_capex_df
    .sort_values(["year", "filed"])
    .drop_duplicates(subset=["year"], keep="last")
    [["year", "capex_billions"]]
    .rename(columns={
        "capex_billions": "capital_expenditures"
    })
)

# Remove any existing CapEx column to make reruns safe
financial_wide = financial_wide.drop(
    columns=["capital_expenditures"],
    errors="ignore"
)

# Merge the CapEx data into our financial table
financial_wide = financial_wide.merge(
    capex_clean,
    on="year",
    how="left",
    validate="one_to_one"
)

financial_wide = (
    financial_wide
    .sort_values("year")
    .reset_index(drop=True)
)

display(financial_wide.round(3))


In [ ]:

# Cell 16 — Complete financial performance analysis

financial_analysis = financial_wide.copy()

financial_analysis["revenue_growth_pct"] = (
    financial_analysis["revenue"].pct_change() * 100
)

financial_analysis["operating_margin_pct"] = (
    financial_analysis["operating_income"]
    / financial_analysis["revenue"] * 100
)

financial_analysis["net_profit_margin_pct"] = (
    financial_analysis["net_income"]
    / financial_analysis["revenue"] * 100
)

financial_analysis["free_cash_flow"] = (
    financial_analysis["operating_cash_flow"]
    - financial_analysis["capital_expenditures"]
)

financial_analysis["fcf_margin_pct"] = (
    financial_analysis["free_cash_flow"]
    / financial_analysis["revenue"] * 100
)

financial_analysis["fcf_conversion_pct"] = (
    financial_analysis["free_cash_flow"]
    / financial_analysis["net_income"] * 100
)

# Select a concise reporting table
report_columns = [
    "year",
    "revenue",
    "revenue_growth_pct",
    "operating_margin_pct",
    "net_profit_margin_pct",
    "free_cash_flow",
    "fcf_margin_pct",
    "fcf_conversion_pct"
]

display(
    financial_analysis[report_columns].round(2)
)


In [ ]:

# Cell 17 — Validate complete financial dataset

expected_years = [2021, 2022, 2023, 2024, 2025]

expected_columns = [
    "revenue",
    "operating_income",
    "net_income",
    "operating_cash_flow",
    "capital_expenditures",
    "total_assets",
    "total_liabilities"
]

assert financial_wide["year"].tolist() == expected_years, (
    "Fiscal year coverage is incomplete or out of order."
)

assert financial_wide["year"].is_unique, (
    "Duplicate fiscal years detected."
)

missing_columns = [
    col for col in expected_columns
    if col not in financial_wide.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

missing_values = financial_wide[
    expected_columns
].isna().sum()

assert missing_values.sum() == 0, (
    f"Missing values detected:\n{missing_values}"
)

assert (financial_wide["revenue"] > 0).all()
assert (financial_wide["total_assets"] > 0).all()
assert (financial_wide["capital_expenditures"] >= 0).all()

print("PASS: All five fiscal years are present.")
print("PASS: No duplicate years.")
print("PASS: All seven financial metrics are present.")
print("PASS: No missing values.")
print("PASS: Basic financial checks passed.")


In [ ]:

# Cell 18 — Export financial datasets

financial_wide.to_csv(
    "lockheed_financials_2021_2025.csv",
    index=False
)

financial_analysis.to_csv(
    "lockheed_financial_analysis_2021_2025.csv",
    index=False
)

print("Financial datasets saved successfully.")


In [ ]:

# Cell 19 — Operating cost and margin analysis

cost_analysis = financial_analysis.copy()

# Implied total operating costs and expenses
cost_analysis["operating_costs"] = (
    cost_analysis["revenue"]
    - cost_analysis["operating_income"]
)

# Percentage of revenue consumed by operating costs
cost_analysis["operating_cost_ratio_pct"] = (
    cost_analysis["operating_costs"]
    / cost_analysis["revenue"] * 100
)

# Annual change in operating costs
cost_analysis["operating_cost_growth_pct"] = (
    cost_analysis["operating_costs"]
    .pct_change() * 100
)

# Annual change in operating income
cost_analysis["operating_income_growth_pct"] = (
    cost_analysis["operating_income"]
    .pct_change() * 100
)

display(
    cost_analysis[
        [
            "year",
            "revenue",
            "operating_costs",
            "operating_cost_ratio_pct",
            "revenue_growth_pct",
            "operating_cost_growth_pct",
            "operating_income_growth_pct"
        ]
    ].round(2)
)


In [ ]:

# Cell 20 — Identify largest margin deterioration

cost_analysis["operating_margin_change_pp"] = (
    cost_analysis["operating_margin_pct"].diff()
)

largest_decline = (
    cost_analysis
    .dropna(subset=["operating_margin_change_pp"])
    .sort_values("operating_margin_change_pp")
    .iloc[0]
)

print(
    "Largest year-over-year operating margin decline:",
    int(largest_decline["year"])
)

print(
    "Margin change:",
    f'{largest_decline["operating_margin_change_pp"]:.2f}',
    "percentage points"
)

display(
    cost_analysis[
        [
            "year",
            "operating_margin_pct",
            "operating_margin_change_pp"
        ]
    ].round(2)
)


In [ ]:

# Cell 21 — FY2024 disclosed classified program losses
# Source: Lockheed Martin FY2024 Form 10-K
# Units: USD billions

program_losses = pd.DataFrame({
    "segment": [
        "Aeronautics",
        "Missiles and Fire Control"
    ],
    "loss_billions": [0.555, 1.410],
    "year": [2024, 2024]
})

total_program_losses = program_losses[
    "loss_billions"
].sum()

operating_profit_decline = (
    financial_wide.loc[
        financial_wide["year"] == 2023,
        "operating_income"
    ].iloc[0]
    -
    financial_wide.loc[
        financial_wide["year"] == 2024,
        "operating_income"
    ].iloc[0]
)

print("2024 disclosed segment-level program losses:",
      f"${total_program_losses:.3f}B")

print("2023–2024 consolidated operating profit decline:",
      f"${operating_profit_decline:.3f}B")

display(program_losses)


In [ ]:

# Cell 22 — Lockheed Martin segment financial performance
# Source: FY2024 Form 10-K, segment results
# All financial values in USD billions

segment_data = pd.DataFrame({
    "segment": [
        "Aeronautics",
        "Missiles and Fire Control",
        "Rotary and Mission Systems",
        "Space"
    ],
    "revenue_2023": [27.474, 11.253, 16.239, 12.605],
    "revenue_2024": [28.618, 12.682, 17.264, 12.479],
    "operating_profit_2023": [2.825, 1.541, 1.865, 1.158],
    "operating_profit_2024": [2.523, 0.413, 1.921, 1.226]
})

# Calculate revenue growth
segment_data["revenue_growth_pct"] = (
    (segment_data["revenue_2024"]
     / segment_data["revenue_2023"] - 1) * 100
)

# Calculate operating profit change
segment_data["profit_change_billions"] = (
    segment_data["operating_profit_2024"]
    - segment_data["operating_profit_2023"]
)

# Calculate operating margins
for year in [2023, 2024]:
    segment_data[f"margin_{year}_pct"] = (
        segment_data[f"operating_profit_{year}"]
        / segment_data[f"revenue_{year}"] * 100
    )

segment_data["margin_change_pp"] = (
    segment_data["margin_2024_pct"]
    - segment_data["margin_2023_pct"]
)

display(segment_data.round(2))


In [ ]:

# Cell 23 — Reconcile segment and consolidated profit
# USD billions

reconciliation = pd.DataFrame({
    "year": [2023, 2024],
    "unallocated_net": [1.118, 0.930]
})

for year in [2023, 2024]:
    segment_total = segment_data[
        f"operating_profit_{year}"
    ].sum()

    unallocated = reconciliation.loc[
        reconciliation["year"] == year,
        "unallocated_net"
    ].iloc[0]

    calculated_total = segment_total + unallocated

    reported_total = financial_wide.loc[
        financial_wide["year"] == year,
        "operating_income"
    ].iloc[0]

    difference = calculated_total - reported_total

    print(f"\nFY{year}")
    print(f"Segment operating profit: ${segment_total:.3f}B")
    print(f"Net unallocated items:    ${unallocated:.3f}B")
    print(f"Consolidated profit:      ${calculated_total:.3f}B")
    print(f"Reconciliation difference: ${difference:.6f}B")

    assert abs(difference) < 0.000001, (
        f"Reconciliation failed for {year}"
    )

print("\nPASS: Both fiscal years reconcile.")


In [ ]:

# Cell 24 — Segment contribution to operating profit changes

segment_contribution = segment_data[
    [
        "segment",
        "profit_change_billions",
        "revenue_growth_pct",
        "margin_change_pp"
    ]
].copy()

total_segment_change = (
    segment_contribution["profit_change_billions"].sum()
)

segment_contribution["contribution_to_net_change_pct"] = (
    segment_contribution["profit_change_billions"]
    / total_segment_change * 100
)

segment_contribution = (
    segment_contribution
    .sort_values("profit_change_billions")
    .reset_index(drop=True)
)

print(
    "Total change in segment operating profit:",
    f"${total_segment_change:.3f}B"
)

display(segment_contribution.round(2))


In [ ]:

# Cell 25 — Operating profit change bridge
# All values in USD billions

segment_profit_change = (
    segment_data["profit_change_billions"].sum()
)

unallocated_change = 0.930 - 1.118

consolidated_change = (
    financial_wide.loc[
        financial_wide["year"] == 2024,
        "operating_income"
    ].iloc[0]
    -
    financial_wide.loc[
        financial_wide["year"] == 2023,
        "operating_income"
    ].iloc[0]
)

bridge = pd.DataFrame({
    "component": [
        "Business segment operating profit",
        "Net unallocated items",
        "Consolidated operating profit"
    ],
    "change_billions": [
        segment_profit_change,
        unallocated_change,
        consolidated_change
    ]
})

display(bridge.round(3))

assert abs(
    segment_profit_change
    + unallocated_change
    - consolidated_change
) < 0.000001

print("PASS: Operating profit change reconciles.")


In [ ]:

# Cell 26 — Segment operating margin comparison

import matplotlib.pyplot as plt
import numpy as np

segments = segment_data["segment"]
x = np.arange(len(segments))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))

ax.bar(
    x - width / 2,
    segment_data["margin_2023_pct"],
    width,
    label="2023"
)

ax.bar(
    x + width / 2,
    segment_data["margin_2024_pct"],
    width,
    label="2024"
)

ax.set_title(
    "Lockheed Martin: Operating Margin by Segment"
)
ax.set_ylabel("Operating Margin (%)")
ax.set_xticks(x)
ax.set_xticklabels(segments, rotation=15, ha="right")
ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:

# Cell 27 — Export segment analysis and profit bridge

segment_data.to_csv(
    "lockheed_segment_analysis_2023_2024.csv",
    index=False
)

segment_contribution.to_csv(
    "lockheed_segment_contributions_2024.csv",
    index=False
)

bridge.to_csv(
    "lockheed_operating_profit_bridge_2024.csv",
    index=False
)

print("Saved three segment analysis datasets.")

print("\nSegment analysis:")
print(segment_data.shape)

print("\nSegment contributions:")
print(segment_contribution.shape)

print("\nOperating profit bridge:")
print(bridge.shape)


## Peer SEC extraction and data standardization

Retrieve financial observations for Lockheed Martin, Northrop Grumman, and General Dynamics, standardize FY2021–2025 metrics, and retain source metadata.

In [ ]:

# Cell 28 — Define peer companies

companies = {
    "LMT": {
        "name": "Lockheed Martin",
        "cik": "0000936468"
    },
    "NOC": {
        "name": "Northrop Grumman",
        "cik": "0001133421"
    },
    "GD": {
        "name": "General Dynamics",
        "cik": "0000040533"
    }
}

for ticker, info in companies.items():
    print(
        ticker,
        "|",
        info["name"],
        "| CIK:",
        info["cik"]
    )


In [ ]:

# Cell 29 — Retrieve SEC Company Facts for all peers

import requests
import time

headers = {
    "User-Agent": SEC_USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}

company_facts = {}

for ticker, info in companies.items():

    url = (
        "https://data.sec.gov/api/xbrl/companyfacts/"
        f"CIK{info['cik']}.json"
    )

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    company_facts[ticker] = response.json()

    print(
        f"{ticker}:",
        company_facts[ticker]["entityName"],
        "| US-GAAP concepts:",
        len(company_facts[ticker]["facts"].get("us-gaap", {}))
    )

    time.sleep(0.2)

print("\nAll company facts retrieved successfully.")


In [ ]:

# Cell 30 — Compare SEC financial concepts across companies

candidate_tags = {
    "revenue": [
        "Revenues",
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "SalesRevenueNet"
    ],
    "operating_income": [
        "OperatingIncomeLoss"
    ],
    "net_income": [
        "NetIncomeLoss"
    ],
    "operating_cash_flow": [
        "NetCashProvidedByUsedInOperatingActivities"
    ],
    "capital_expenditures": [
        "PaymentsToAcquireProductiveAssets",
        "PaymentsToAcquirePropertyPlantAndEquipment"
    ],
    "total_assets": [
        "Assets"
    ],
    "total_liabilities": [
        "Liabilities"
    ]
}

tag_records = []

for ticker, facts in company_facts.items():

    gaap_facts = facts["facts"].get("us-gaap", {})

    for metric, tags in candidate_tags.items():

        for tag in tags:

            if tag not in gaap_facts:
                continue

            observations = (
                gaap_facts[tag]["units"].get("USD", [])
            )

            annual_records = []

            for obs in observations:

                if obs.get("form") != "10-K":
                    continue

                if "end" not in obs:
                    continue

                year = int(obs["end"][:4])

                if not 2021 <= year <= 2025:
                    continue

                if metric not in [
                    "total_assets",
                    "total_liabilities"
                ]:
                    if "start" not in obs:
                        continue

                    duration = (
                        pd.to_datetime(obs["end"])
                        - pd.to_datetime(obs["start"])
                    ).days

                    if not 350 <= duration <= 380:
                        continue

                else:
                    if "start" in obs:
                        continue

                annual_records.append(obs)

            covered_years = sorted({
                int(obs["end"][:4])
                for obs in annual_records
            })

            tag_records.append({
                "ticker": ticker,
                "metric": metric,
                "tag": tag,
                "annual_observations": len(annual_records),
                "years_covered": covered_years
            })

tag_comparison = pd.DataFrame(tag_records)

display(
    tag_comparison.sort_values(
        ["ticker", "metric", "tag"]
    )
)


In [ ]:

# Cell 31 — Summarize SEC tag coverage

coverage_summary = (
    tag_comparison[
        tag_comparison["annual_observations"] > 0
    ]
    .copy()
)

coverage_summary["year_count"] = (
    coverage_summary["years_covered"]
    .apply(len)
)

coverage_summary["complete_coverage"] = (
    coverage_summary["year_count"] == 5
)

display(
    coverage_summary[
        [
            "ticker",
            "metric",
            "tag",
            "year_count",
            "complete_coverage"
        ]
    ].sort_values(
        ["ticker", "metric", "tag"]
    )
)


In [ ]:

# Cell 32 — Compare General Dynamics revenue concepts

gd_gaap = company_facts["GD"]["facts"]["us-gaap"]

revenue_candidates = [
    "Revenues",
    "RevenueFromContractWithCustomerExcludingAssessedTax"
]

gd_revenue_records = []

for tag in revenue_candidates:

    observations = gd_gaap[tag]["units"].get("USD", [])

    for obs in observations:

        if obs.get("form") != "10-K":
            continue

        if "start" not in obs or "end" not in obs:
            continue

        year = int(obs["end"][:4])

        if not 2021 <= year <= 2025:
            continue

        duration = (
            pd.to_datetime(obs["end"])
            - pd.to_datetime(obs["start"])
        ).days

        if not 350 <= duration <= 380:
            continue

        gd_revenue_records.append({
            "year": year,
            "tag": tag,
            "revenue_billions": obs["val"] / 1e9,
            "filed": obs.get("filed")
        })

gd_revenue_df = pd.DataFrame(gd_revenue_records)

gd_revenue_comparison = (
    gd_revenue_df
    .sort_values(["year", "tag", "filed"])
    .drop_duplicates(
        subset=["year", "tag"],
        keep="last"
    )
    .pivot(
        index="year",
        columns="tag",
        values="revenue_billions"
    )
)

gd_revenue_comparison["difference_billions"] = (
    gd_revenue_comparison["Revenues"]
    - gd_revenue_comparison[
        "RevenueFromContractWithCustomerExcludingAssessedTax"
    ]
)

display(gd_revenue_comparison.round(3))


In [ ]:

# Cell 33 — Investigate General Dynamics liabilities

gd_gaap = company_facts["GD"]["facts"]["us-gaap"]

candidate_balance_sheet_tags = [
    "Liabilities",
    "LiabilitiesAndStockholdersEquity",
    "StockholdersEquity",
    "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    "Assets"
]

for tag in candidate_balance_sheet_tags:

    print(f"\n--- {tag} ---")

    if tag not in gd_gaap:
        print("Tag not available")
        continue

    observations = gd_gaap[tag]["units"].get("USD", [])

    annual_records = []

    for obs in observations:

        if obs.get("form") != "10-K":
            continue

        if "end" not in obs or "start" in obs:
            continue

        year = int(obs["end"][:4])

        if not 2021 <= year <= 2025:
            continue

        annual_records.append({
            "year": year,
            "value_billions": obs["val"] / 1e9,
            "filed": obs.get("filed")
        })

    if annual_records:

        df = pd.DataFrame(annual_records)

        df = (
            df.sort_values(["year", "filed"])
            .drop_duplicates("year", keep="last")
        )

        display(df.round(3))

    else:
        print("No qualifying observations.")


In [ ]:

# Cell 34 — Company-specific SEC financial tags

company_metric_tags = {
    "LMT": {
        "revenue": "Revenues",
        "operating_income": "OperatingIncomeLoss",
        "net_income": "NetIncomeLoss",
        "operating_cash_flow": "NetCashProvidedByUsedInOperatingActivities",
        "capital_expenditures": "PaymentsToAcquireProductiveAssets",
        "total_assets": "Assets",
        "total_liabilities": "Liabilities"
    },
    "NOC": {
        "revenue": "Revenues",
        "operating_income": "OperatingIncomeLoss",
        "net_income": "NetIncomeLoss",
        "operating_cash_flow": "NetCashProvidedByUsedInOperatingActivities",
        "capital_expenditures": "PaymentsToAcquirePropertyPlantAndEquipment",
        "total_assets": "Assets",
        "total_liabilities": "Liabilities"
    },
    "GD": {
        "revenue": "Revenues",
        "operating_income": "OperatingIncomeLoss",
        "net_income": "NetIncomeLoss",
        "operating_cash_flow": "NetCashProvidedByUsedInOperatingActivities",
        "capital_expenditures": "PaymentsToAcquirePropertyPlantAndEquipment",
        "total_assets": "Assets",
        "total_liabilities": None
    }
}

print("Company-specific financial tags configured.")

for ticker, tags in company_metric_tags.items():
    print(f"\n{ticker}:")
    for metric, tag in tags.items():
        print(f"  {metric}: {tag}")


In [ ]:

# Cell 35 — Reusable SEC financial extraction function

def extract_company_financials(
    ticker,
    company_facts,
    metric_tags,
    start_year=2021,
    end_year=2025
):
    gaap = company_facts[ticker]["facts"]["us-gaap"]

    instant_metrics = {
        "total_assets",
        "total_liabilities"
    }

    records = []

    for metric, tag in metric_tags[ticker].items():

        if tag is None:
            continue

        if tag not in gaap:
            print(f"WARNING: {ticker} missing tag {tag}")
            continue

        observations = gaap[tag]["units"].get("USD", [])

        for obs in observations:

            if obs.get("form") != "10-K":
                continue

            period_end = obs.get("end")

            if not period_end:
                continue

            year = int(period_end[:4])

            if not start_year <= year <= end_year:
                continue

            if metric in instant_metrics:
                if "start" in obs:
                    continue
            else:
                if "start" not in obs:
                    continue

                duration = (
                    pd.to_datetime(obs["end"])
                    - pd.to_datetime(obs["start"])
                ).days

                if not 350 <= duration <= 380:
                    continue

            records.append({
                "ticker": ticker,
                "year": year,
                "metric": metric,
                "value_billions": obs["val"] / 1e9,
                "source_tag": tag,
                "period_start": obs.get("start"),
                "period_end": period_end,
                "filed": obs.get("filed"),
                "accession": obs.get("accn")
            })

    df = pd.DataFrame(records)

    if df.empty:
        raise ValueError(
            f"No financial observations found for {ticker}"
        )

    # Select the latest filed observation for each metric/year
    df = (
        df.sort_values(
            ["ticker", "year", "metric", "filed"]
        )
        .drop_duplicates(
            subset=["ticker", "year", "metric"],
            keep="last"
        )
        .reset_index(drop=True)
    )

    return df

print("Reusable extraction function created.")


In [ ]:

# Cell 36 — Extract standardized financial data

all_financial_records = []

for ticker in companies:

    company_df = extract_company_financials(
        ticker=ticker,
        company_facts=company_facts,
        metric_tags=company_metric_tags
    )

    all_financial_records.append(company_df)

    print(
        f"{ticker}: extracted",
        len(company_df),
        "financial observations"
    )

financial_audit = pd.concat(
    all_financial_records,
    ignore_index=True
)

print(
    "\nTotal selected observations:",
    len(financial_audit)
)

display(financial_audit.head(10))


In [ ]:

# Cell 37 — Create standardized peer financial table

peer_financials = (
    financial_audit
    .pivot(
        index=["ticker", "year"],
        columns="metric",
        values="value_billions"
    )
    .reset_index()
)

peer_financials.columns.name = None

# Derive General Dynamics liabilities
gd_gaap = company_facts["GD"]["facts"]["us-gaap"]

equity_records = []

for obs in gd_gaap["StockholdersEquity"]["units"]["USD"]:

    if obs.get("form") != "10-K":
        continue

    if "start" in obs or "end" not in obs:
        continue

    year = int(obs["end"][:4])

    if not 2021 <= year <= 2025:
        continue

    equity_records.append({
        "year": year,
        "equity_billions": obs["val"] / 1e9,
        "filed": obs.get("filed")
    })

gd_equity = (
    pd.DataFrame(equity_records)
    .sort_values(["year", "filed"])
    .drop_duplicates("year", keep="last")
    [["year", "equity_billions"]]
)

gd_mask = peer_financials["ticker"] == "GD"

peer_financials = peer_financials.merge(
    gd_equity,
    on="year",
    how="left",
    validate="many_to_one"
)

peer_financials.loc[
    gd_mask,
    "total_liabilities"
] = (
    peer_financials.loc[gd_mask, "total_assets"]
    - peer_financials.loc[gd_mask, "equity_billions"]
)

peer_financials = peer_financials.drop(
    columns=["equity_billions"]
)

peer_financials = (
    peer_financials
    .sort_values(["ticker", "year"])
    .reset_index(drop=True)
)

display(peer_financials.round(3))


## Data validation and peer benchmarking

Validate historical records and calculate comparative financial ratios and peer scorecards.

In [ ]:

# Cell 38 — Validate standardized peer dataset

expected_tickers = {"LMT", "NOC", "GD"}
expected_years = set(range(2021, 2026))

financial_metrics = [
    "revenue",
    "operating_income",
    "net_income",
    "operating_cash_flow",
    "capital_expenditures",
    "total_assets",
    "total_liabilities"
]

# Check number of records
assert len(peer_financials) == 15, (
    "Expected exactly 15 company-year records."
)

# Check duplicate records
assert not peer_financials.duplicated(
    subset=["ticker", "year"]
).any(), "Duplicate company-year records detected."

# Check company and year coverage
assert set(peer_financials["ticker"]) == expected_tickers

for ticker in expected_tickers:
    years = set(
        peer_financials.loc[
            peer_financials["ticker"] == ticker,
            "year"
        ]
    )

    assert years == expected_years, (
        f"Incomplete fiscal year coverage for {ticker}"
    )

# Check missing values
assert not peer_financials[
    financial_metrics
].isna().any().any(), (
    "Missing financial values detected."
)

# Basic financial plausibility
assert (peer_financials["revenue"] > 0).all()
assert (peer_financials["total_assets"] > 0).all()
assert (peer_financials["capital_expenditures"] >= 0).all()

assert (
    peer_financials["total_liabilities"]
    <= peer_financials["total_assets"]
).all(), "Liabilities exceed assets; investigate."

print("PASS: 15 company-year records.")
print("PASS: No duplicate records.")
print("PASS: Complete FY2021–FY2025 coverage.")
print("PASS: No missing financial metrics.")
print("PASS: Basic financial plausibility checks.")


In [ ]:

# Cell 39 — Calculate financial performance metrics

peer_analysis = (
    peer_financials
    .sort_values(["ticker", "year"])
    .reset_index(drop=True)
    .copy()
)

# Revenue growth, calculated separately for each company
peer_analysis["revenue_growth_pct"] = (
    peer_analysis
    .groupby("ticker")["revenue"]
    .pct_change() * 100
)

# Operating profit margin
peer_analysis["operating_margin_pct"] = (
    peer_analysis["operating_income"]
    / peer_analysis["revenue"] * 100
)

# Net profit margin
peer_analysis["net_margin_pct"] = (
    peer_analysis["net_income"]
    / peer_analysis["revenue"] * 100
)

# Free cash flow
peer_analysis["free_cash_flow"] = (
    peer_analysis["operating_cash_flow"]
    - peer_analysis["capital_expenditures"]
)

# Free cash flow margin
peer_analysis["fcf_margin_pct"] = (
    peer_analysis["free_cash_flow"]
    / peer_analysis["revenue"] * 100
)

# Total liabilities relative to total assets
peer_analysis["liabilities_to_assets_pct"] = (
    peer_analysis["total_liabilities"]
    / peer_analysis["total_assets"] * 100
)

display(peer_analysis.round(2))


In [ ]:

# Cell 40 — FY2025 peer benchmarking

peer_2025 = (
    peer_analysis[
        peer_analysis["year"] == 2025
    ]
    .copy()
)

comparison_columns = [
    "ticker",
    "revenue",
    "revenue_growth_pct",
    "operating_margin_pct",
    "net_margin_pct",
    "free_cash_flow",
    "fcf_margin_pct",
    "liabilities_to_assets_pct"
]

peer_2025 = peer_2025[
    comparison_columns
].sort_values(
    "revenue",
    ascending=False
)

display(peer_2025.round(2))


In [ ]:

# Cell 41 — Five-year financial performance scorecard

peer_scorecard_records = []

for ticker, group in peer_analysis.groupby("ticker"):

    group = group.sort_values("year")

    first = group.iloc[0]
    last = group.iloc[-1]

    # Four annual growth intervals: 2021 to 2025
    years = int(last["year"] - first["year"])

    revenue_cagr = (
        (last["revenue"] / first["revenue"])
        ** (1 / years) - 1
    ) * 100

    peer_scorecard_records.append({
        "ticker": ticker,
        "revenue_cagr_pct": revenue_cagr,
        "avg_operating_margin_pct": (
            group["operating_margin_pct"].mean()
        ),
        "avg_fcf_margin_pct": (
            group["fcf_margin_pct"].mean()
        ),
        "total_fcf_billions": (
            group["free_cash_flow"].sum()
        ),
        "operating_margin_change_pp": (
            last["operating_margin_pct"]
            - first["operating_margin_pct"]
        ),
        "revenue_growth_total_pct": (
            (last["revenue"] / first["revenue"] - 1)
            * 100
        )
    })

peer_scorecard = pd.DataFrame(
    peer_scorecard_records
).sort_values(
    "revenue_cagr_pct",
    ascending=False
).reset_index(drop=True)

display(peer_scorecard.round(2))


In [ ]:

# Cell 42 — Operating margin stability

margin_stability = (
    peer_analysis
    .groupby("ticker")
    .agg(
        min_operating_margin_pct=(
            "operating_margin_pct", "min"
        ),
        max_operating_margin_pct=(
            "operating_margin_pct", "max"
        ),
        avg_operating_margin_pct=(
            "operating_margin_pct", "mean"
        ),
        operating_margin_std_pp=(
            "operating_margin_pct", "std"
        )
    )
    .reset_index()
)

margin_stability["margin_range_pp"] = (
    margin_stability["max_operating_margin_pct"]
    - margin_stability["min_operating_margin_pct"]
)

display(
    margin_stability.sort_values(
        "operating_margin_std_pp"
    ).round(2)
)


In [ ]:

# Cell 43 — Export peer benchmarking datasets

peer_financials.to_csv(
    "defense_peer_financials_2021_2025.csv",
    index=False
)

financial_audit.to_csv(
    "defense_peer_sec_audit.csv",
    index=False
)

peer_analysis.to_csv(
    "defense_peer_financial_ratios.csv",
    index=False
)

peer_scorecard.to_csv(
    "defense_peer_scorecard.csv",
    index=False
)

margin_stability.to_csv(
    "defense_peer_margin_stability.csv",
    index=False
)

print("Saved five peer benchmarking datasets.")


## SQLite database and SQL analysis

Build financial and SEC source tables, calculate SQL financial ratios, and create peer ranking views.

In [ ]:

# Cell 44 — Create SQLite database

import sqlite3

db_path = "defense_financial_intelligence.db"

conn = sqlite3.connect(db_path)

conn.execute("PRAGMA foreign_keys = ON;")

print("SQLite database created:", db_path)


In [ ]:

# Cell 45 — Create companies and financials tables

conn.executescript("""
CREATE TABLE IF NOT EXISTS companies (
    ticker TEXT PRIMARY KEY,
    company_name TEXT NOT NULL,
    cik TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS financials (
    ticker TEXT NOT NULL,
    year INTEGER NOT NULL,

    revenue REAL NOT NULL,
    operating_income REAL NOT NULL,
    net_income REAL NOT NULL,
    operating_cash_flow REAL NOT NULL,
    capital_expenditures REAL NOT NULL,
    free_cash_flow REAL NOT NULL,
    total_assets REAL NOT NULL,
    total_liabilities REAL NOT NULL,

    PRIMARY KEY (ticker, year),

    FOREIGN KEY (ticker)
        REFERENCES companies(ticker),

    CHECK (year BETWEEN 2021 AND 2025),
    CHECK (revenue > 0),
    CHECK (total_assets > 0),
    CHECK (capital_expenditures >= 0)
);
""")

conn.commit()

print("Created companies and financials tables.")


In [ ]:

# Cell 46 — Insert companies into SQLite

company_rows = [
    (
        ticker,
        info["name"],
        info["cik"]
    )
    for ticker, info in companies.items()
]

conn.executemany("""
INSERT OR REPLACE INTO companies (
    ticker,
    company_name,
    cik
)
VALUES (?, ?, ?)
""", company_rows)

conn.commit()

companies_sql = pd.read_sql_query(
    "SELECT * FROM companies ORDER BY ticker;",
    conn
)

display(companies_sql)


In [ ]:

# Cell 47 — Insert standardized financial records

financial_columns = [
    "ticker",
    "year",
    "revenue",
    "operating_income",
    "net_income",
    "operating_cash_flow",
    "capital_expenditures",
    "free_cash_flow",
    "total_assets",
    "total_liabilities"
]

financial_rows = list(
    peer_analysis[financial_columns]
    .itertuples(index=False, name=None)
)

conn.executemany("""
INSERT OR REPLACE INTO financials (
    ticker,
    year,
    revenue,
    operating_income,
    net_income,
    operating_cash_flow,
    capital_expenditures,
    free_cash_flow,
    total_assets,
    total_liabilities
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", financial_rows)

conn.commit()

print(
    "Financial records loaded:",
    conn.execute(
        "SELECT COUNT(*) FROM financials"
    ).fetchone()[0]
)


In [ ]:

# Cell 48 — Verify financial database

verification_query = """
SELECT
    c.company_name,
    f.ticker,
    COUNT(*) AS years_reported,
    MIN(f.year) AS first_year,
    MAX(f.year) AS last_year,
    ROUND(SUM(f.free_cash_flow), 2) AS total_fcf_billions
FROM financials AS f
JOIN companies AS c
    ON f.ticker = c.ticker
GROUP BY
    c.company_name,
    f.ticker
ORDER BY total_fcf_billions DESC;
"""

verification_results = pd.read_sql_query(
    verification_query,
    conn
)

display(verification_results)


In [ ]:

# Cell 49 — Create SEC source audit table

conn.executescript("""
CREATE TABLE IF NOT EXISTS sec_sources (
    ticker TEXT NOT NULL,
    year INTEGER NOT NULL,
    metric TEXT NOT NULL,

    value_billions REAL NOT NULL,
    source_tag TEXT,
    period_start TEXT,
    period_end TEXT,
    filed TEXT,
    accession TEXT,

    source_type TEXT NOT NULL,

    PRIMARY KEY (ticker, year, metric),

    FOREIGN KEY (ticker, year)
        REFERENCES financials(ticker, year),

    CHECK (
        source_type IN (
            'SEC_REPORTED',
            'DERIVED'
        )
    )
);
""")

conn.commit()

print("SEC source audit table created.")


In [ ]:

# Cell 50 — Load SEC-reported financial sources

source_columns = [
    "ticker",
    "year",
    "metric",
    "value_billions",
    "source_tag",
    "period_start",
    "period_end",
    "filed",
    "accession"
]

reported_sources = financial_audit[
    source_columns
].copy()

reported_sources["source_type"] = "SEC_REPORTED"

source_rows = list(
    reported_sources.itertuples(
        index=False,
        name=None
    )
)

conn.executemany("""
INSERT OR REPLACE INTO sec_sources (
    ticker,
    year,
    metric,
    value_billions,
    source_tag,
    period_start,
    period_end,
    filed,
    accession,
    source_type
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", source_rows)

conn.commit()

print(
    "SEC-reported source records:",
    conn.execute("""
        SELECT COUNT(*)
        FROM sec_sources
        WHERE source_type = 'SEC_REPORTED'
    """).fetchone()[0]
)


In [ ]:

# Cell 51 — Add derived General Dynamics liabilities

gd_liabilities = peer_financials[
    peer_financials["ticker"] == "GD"
][
    ["ticker", "year", "total_liabilities"]
].copy()

gd_liabilities["metric"] = "total_liabilities"

gd_liabilities["source_tag"] = (
    "Assets - StockholdersEquity"
)

gd_liabilities["source_type"] = "DERIVED"

derived_rows = [
    (
        row.ticker,
        int(row.year),
        row.metric,
        float(row.total_liabilities),
        row.source_tag,
        None,
        None,
        None,
        None,
        row.source_type
    )
    for row in gd_liabilities.itertuples(index=False)
]

conn.executemany("""
INSERT OR REPLACE INTO sec_sources (
    ticker,
    year,
    metric,
    value_billions,
    source_tag,
    period_start,
    period_end,
    filed,
    accession,
    source_type
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", derived_rows)

conn.commit()

print(
    "Total source records:",
    conn.execute(
        "SELECT COUNT(*) FROM sec_sources"
    ).fetchone()[0]
)


In [ ]:

# Cell 52 — Verify SEC source coverage

source_verification_query = """
SELECT
    ticker,
    source_type,
    COUNT(*) AS source_records,
    COUNT(DISTINCT year) AS years_covered
FROM sec_sources
GROUP BY
    ticker,
    source_type
ORDER BY
    ticker,
    source_type;
"""

source_verification = pd.read_sql_query(
    source_verification_query,
    conn
)

display(source_verification)

assert len(pd.read_sql_query(
    "SELECT * FROM sec_sources", conn
)) == 105

print("\nPASS: All 105 financial source records are present.")


In [ ]:

# Cell 53 — Calculate year-over-year revenue growth in SQL

revenue_growth_query = """
WITH revenue_history AS (
    SELECT
        ticker,
        year,
        revenue,

        LAG(revenue) OVER (
            PARTITION BY ticker
            ORDER BY year
        ) AS previous_year_revenue

    FROM financials
)

SELECT
    ticker,
    year,
    ROUND(revenue, 3) AS revenue_billions,
    ROUND(previous_year_revenue, 3)
        AS previous_year_revenue_billions,

    ROUND(
        (
            revenue / NULLIF(previous_year_revenue, 0)
            - 1
        ) * 100,
        2
    ) AS revenue_growth_pct

FROM revenue_history

ORDER BY ticker, year;
"""

sql_revenue_growth = pd.read_sql_query(
    revenue_growth_query,
    conn
)

display(sql_revenue_growth)


In [ ]:

# Cell 54 — Validate SQL revenue growth against pandas

growth_validation = sql_revenue_growth.merge(
    peer_analysis[
        ["ticker", "year", "revenue_growth_pct"]
    ],
    on=["ticker", "year"],
    how="inner",
    validate="one_to_one",
    suffixes=("_sql", "_pandas")
)

growth_validation["absolute_difference"] = (
    growth_validation["revenue_growth_pct_sql"]
    - growth_validation["revenue_growth_pct_pandas"]
).abs()

# SQL output was rounded to two decimals,
# so allow a 0.01 percentage-point difference.
assert (
    growth_validation["absolute_difference"]
    .dropna()
    <= 0.01
).all(), "SQL and pandas revenue growth disagree."

assert len(growth_validation) == 15

print("PASS: SQL revenue growth matches pandas.")
print("Validated company-year records:", len(growth_validation))

display(
    growth_validation[
        [
            "ticker",
            "year",
            "revenue_growth_pct_sql",
            "revenue_growth_pct_pandas",
            "absolute_difference"
        ]
    ].round(3)
)


In [ ]:

# Cell 55 — Rank operating margins by year

margin_ranking_query = """
SELECT
    year,
    ticker,
    ROUND(
        operating_income / NULLIF(revenue, 0) * 100,
        2
    ) AS operating_margin_pct,

    RANK() OVER (
        PARTITION BY year
        ORDER BY
            operating_income / NULLIF(revenue, 0) DESC
    ) AS operating_margin_rank

FROM financials

ORDER BY year, operating_margin_rank;
"""

sql_margin_rankings = pd.read_sql_query(
    margin_ranking_query,
    conn
)

display(sql_margin_rankings)


In [ ]:

# Cell 56 — Rank companies by revenue growth

growth_ranking_query = """
WITH revenue_history AS (
    SELECT
        ticker,
        year,
        revenue,
        LAG(revenue) OVER (
            PARTITION BY ticker
            ORDER BY year
        ) AS previous_revenue
    FROM financials
),

growth_rates AS (
    SELECT
        ticker,
        year,
        (
            revenue / NULLIF(previous_revenue, 0) - 1
        ) * 100 AS revenue_growth_pct
    FROM revenue_history
)

SELECT
    year,
    ticker,
    ROUND(revenue_growth_pct, 2)
        AS revenue_growth_pct,

    RANK() OVER (
        PARTITION BY year
        ORDER BY revenue_growth_pct DESC
    ) AS growth_rank

FROM growth_rates

WHERE revenue_growth_pct IS NOT NULL

ORDER BY year, growth_rank;
"""

sql_growth_rankings = pd.read_sql_query(
    growth_ranking_query,
    conn
)

display(sql_growth_rankings)


In [ ]:

# Cell 57 — Create a reusable SQL financial performance view

conn.executescript("""
DROP VIEW IF EXISTS financial_performance_view;

CREATE VIEW financial_performance_view AS

WITH financial_history AS (
    SELECT
        f.*,
        c.company_name,

        LAG(f.revenue) OVER (
            PARTITION BY f.ticker
            ORDER BY f.year
        ) AS previous_revenue

    FROM financials AS f

    JOIN companies AS c
        ON f.ticker = c.ticker
)

SELECT
    ticker,
    company_name,
    year,

    revenue,
    operating_income,
    net_income,
    operating_cash_flow,
    capital_expenditures,
    free_cash_flow,
    total_assets,
    total_liabilities,

    (revenue / NULLIF(previous_revenue, 0) - 1)
        * 100 AS revenue_growth_pct,

    operating_income / NULLIF(revenue, 0)
        * 100 AS operating_margin_pct,

    net_income / NULLIF(revenue, 0)
        * 100 AS net_margin_pct,

    free_cash_flow / NULLIF(revenue, 0)
        * 100 AS fcf_margin_pct,

    total_liabilities / NULLIF(total_assets, 0)
        * 100 AS liabilities_to_assets_pct

FROM financial_history;
""")

conn.commit()

print("Financial performance SQL view created.")


In [ ]:

# Cell 58 — Query FY2025 financial performance

performance_query = """
SELECT
    ticker,
    ROUND(revenue, 2) AS revenue_billions,
    ROUND(revenue_growth_pct, 2) AS revenue_growth_pct,
    ROUND(operating_margin_pct, 2) AS operating_margin_pct,
    ROUND(net_margin_pct, 2) AS net_margin_pct,
    ROUND(free_cash_flow, 2) AS free_cash_flow_billions,
    ROUND(fcf_margin_pct, 2) AS fcf_margin_pct,
    ROUND(liabilities_to_assets_pct, 2)
        AS liabilities_to_assets_pct

FROM financial_performance_view

WHERE year = 2025

ORDER BY revenue DESC;
"""

sql_performance_2025 = pd.read_sql_query(
    performance_query,
    conn
)

display(sql_performance_2025)


In [ ]:

# Cell 59 — Validate SQL financial ratios against pandas

validation_metrics = [
    "revenue_growth_pct",
    "operating_margin_pct",
    "net_margin_pct",
    "fcf_margin_pct",
    "liabilities_to_assets_pct"
]

sql_metrics = pd.read_sql_query(
    """
    SELECT *
    FROM financial_performance_view
    ORDER BY ticker, year
    """,
    conn
)

comparison = sql_metrics[
    ["ticker", "year"] + validation_metrics
].merge(
    peer_analysis[
        ["ticker", "year"] + validation_metrics
    ],
    on=["ticker", "year"],
    suffixes=("_sql", "_pandas"),
    validate="one_to_one"
)

assert len(comparison) == 15

for metric in validation_metrics:

    sql_col = f"{metric}_sql"
    pandas_col = f"{metric}_pandas"

    differences = (
        comparison[sql_col]
        - comparison[pandas_col]
    ).abs()

    assert (
        differences.dropna() < 1e-8
    ).all(), f"Mismatch detected for {metric}"

    assert (
        comparison[sql_col].isna()
        == comparison[pandas_col].isna()
    ).all(), f"Missing-value mismatch for {metric}"

    print(f"PASS: {metric}")

print("\nAll SQL financial ratios match pandas.")


In [ ]:

# Cell 60 — Create annual peer rankings

conn.executescript("""
DROP VIEW IF EXISTS peer_rankings_view;

CREATE VIEW peer_rankings_view AS

SELECT
    year,
    ticker,
    company_name,

    revenue_growth_pct,
    operating_margin_pct,
    fcf_margin_pct,

    RANK() OVER (
        PARTITION BY year
        ORDER BY revenue_growth_pct DESC
    ) AS revenue_growth_rank,

    RANK() OVER (
        PARTITION BY year
        ORDER BY operating_margin_pct DESC
    ) AS operating_margin_rank,

    RANK() OVER (
        PARTITION BY year
        ORDER BY fcf_margin_pct DESC
    ) AS fcf_margin_rank

FROM financial_performance_view;
""")

conn.commit()

ranking_results = pd.read_sql_query(
    """
    SELECT
        ticker,
        ROUND(revenue_growth_pct, 2) AS growth_pct,
        revenue_growth_rank,
        ROUND(operating_margin_pct, 2) AS operating_margin_pct,
        operating_margin_rank,
        ROUND(fcf_margin_pct, 2) AS fcf_margin_pct,
        fcf_margin_rank

    FROM peer_rankings_view

    WHERE year = 2025

    ORDER BY revenue_growth_rank;
    """,
    conn
)

display(ranking_results)


In [ ]:

# Cell 61 — Export SQL analysis results

sql_performance = pd.read_sql_query(
    """
    SELECT *
    FROM financial_performance_view
    ORDER BY ticker, year
    """,
    conn
)

sql_rankings = pd.read_sql_query(
    """
    SELECT *
    FROM peer_rankings_view
    ORDER BY year, ticker
    """,
    conn
)

sql_performance.to_csv(
    "defense_sql_financial_performance.csv",
    index=False
)

sql_rankings.to_csv(
    "defense_sql_peer_rankings.csv",
    index=False
)

print("Exported SQL financial performance:", sql_performance.shape)
print("Exported SQL peer rankings:", sql_rankings.shape)


In [ ]:

# Cell 62 — Export SQL view definitions

view_definitions = pd.read_sql_query(
    """
    SELECT name, sql
    FROM sqlite_master
    WHERE type = 'view'
      AND name IN (
          'financial_performance_view',
          'peer_rankings_view'
      )
    ORDER BY name
    """,
    conn
)

sql_script = "\n\n".join(
    "DROP VIEW IF EXISTS " + row["name"] + ";\n"
    + row["sql"] + ";"
    for _, row in view_definitions.iterrows()
)

with open(
    "defense_financial_analysis.sql",
    "w",
    encoding="utf-8"
) as file:
    file.write(sql_script + "\n")

print("Saved defense_financial_analysis.sql")
print("Number of SQL views:", len(view_definitions))


In [ ]:

# Cell 63 — Verify exported portfolio files

import os

export_files = [
    "defense_financial_intelligence.db",
    "defense_financial_analysis.sql",
    "defense_sql_financial_performance.csv",
    "defense_sql_peer_rankings.csv"
]

for filename in export_files:

    if os.path.exists(filename):
        size_kb = os.path.getsize(filename) / 1024

        print(
            f"PASS: {filename} ({size_kb:.1f} KB)"
        )

    else:
        print(f"MISSING: {filename}")


## Forecast assumptions and scenarios

Create illustrative FY2026–2027 Bear, Base, and Bull financial projections.

In [ ]:

# Cell 64 — Prepare historical forecasting inputs

forecast_history = peer_analysis[
    [
        "ticker",
        "year",
        "revenue",
        "revenue_growth_pct",
        "operating_income",
        "operating_margin_pct",
        "free_cash_flow",
        "fcf_margin_pct"
    ]
].copy()

forecast_history = (
    forecast_history
    .sort_values(["ticker", "year"])
    .reset_index(drop=True)
)

display(forecast_history.round(2))

print("\nHistorical forecast records:", len(forecast_history))


In [ ]:

# Cell 65 — Calculate recent historical benchmarks

recent_history = forecast_history[
    forecast_history["year"].between(2023, 2025)
].copy()

forecast_benchmarks = (
    recent_history
    .groupby("ticker")
    .agg(
        avg_revenue_growth_pct=(
            "revenue_growth_pct", "mean"
        ),
        avg_operating_margin_pct=(
            "operating_margin_pct", "mean"
        ),
        avg_fcf_margin_pct=(
            "fcf_margin_pct", "mean"
        )
    )
    .reset_index()
)

# Add FY2025 starting values
latest_financials = forecast_history[
    forecast_history["year"] == 2025
][
    ["ticker", "revenue", "operating_income", "free_cash_flow"]
].copy()

forecast_benchmarks = forecast_benchmarks.merge(
    latest_financials,
    on="ticker",
    how="left",
    validate="one_to_one"
)

display(forecast_benchmarks.round(2))


In [ ]:

# Cell 66 — Validate and export forecasting inputs

assert len(forecast_history) == 15
assert len(forecast_benchmarks) == 3

assert not forecast_benchmarks.isna().any().any(), (
    "Missing forecasting benchmark values."
)

forecast_history.to_csv(
    "defense_forecast_history.csv",
    index=False
)

forecast_benchmarks.to_csv(
    "defense_forecast_benchmarks.csv",
    index=False
)

print("PASS: Historical forecasting inputs validated.")
print("Saved defense_forecast_history.csv")
print("Saved defense_forecast_benchmarks.csv")


In [ ]:

# Cell 67 — Create Bear, Base, and Bull assumptions

import pandas as pd

scenario_adjustments = pd.DataFrame({
    "scenario": ["Bear", "Base", "Bull"],
    "growth_adjustment_pp": [-3.0, 0.0, 3.0],
    "operating_margin_adjustment_pp": [-1.0, 0.0, 1.0],
    "fcf_margin_adjustment_pp": [-1.0, 0.0, 1.0]
})

forecast_assumptions = forecast_benchmarks.merge(
    scenario_adjustments,
    how="cross"
)

forecast_assumptions["revenue_growth_pct"] = (
    forecast_assumptions["avg_revenue_growth_pct"]
    + forecast_assumptions["growth_adjustment_pp"]
)

forecast_assumptions["operating_margin_pct"] = (
    forecast_assumptions["avg_operating_margin_pct"]
    + forecast_assumptions["operating_margin_adjustment_pp"]
)

forecast_assumptions["fcf_margin_pct"] = (
    forecast_assumptions["avg_fcf_margin_pct"]
    + forecast_assumptions["fcf_margin_adjustment_pp"]
)

forecast_assumptions = forecast_assumptions[
    [
        "ticker",
        "scenario",
        "revenue",
        "operating_income",
        "free_cash_flow",
        "revenue_growth_pct",
        "operating_margin_pct",
        "fcf_margin_pct"
    ]
]

display(forecast_assumptions.round(2))


In [ ]:

# Cell 68 — Calculate scenario projections

forecast_rows = []

for row in forecast_assumptions.itertuples(index=False):

    previous_revenue = row.revenue

    for year in [2026, 2027]:

        projected_revenue = (
            previous_revenue
            * (1 + row.revenue_growth_pct / 100)
        )

        projected_operating_income = (
            projected_revenue
            * row.operating_margin_pct / 100
        )

        projected_fcf = (
            projected_revenue
            * row.fcf_margin_pct / 100
        )

        forecast_rows.append({
            "ticker": row.ticker,
            "scenario": row.scenario,
            "year": year,
            "revenue": projected_revenue,
            "operating_income": projected_operating_income,
            "free_cash_flow": projected_fcf,
            "revenue_growth_pct": row.revenue_growth_pct,
            "operating_margin_pct": row.operating_margin_pct,
            "fcf_margin_pct": row.fcf_margin_pct
        })

        previous_revenue = projected_revenue

scenario_forecast = pd.DataFrame(forecast_rows)

display(
    scenario_forecast.sort_values(
        ["ticker", "scenario", "year"]
    ).round(2)
)


In [ ]:

# Cell 69 — Summarize FY2027 scenario outcomes

forecast_2027 = scenario_forecast[
    scenario_forecast["year"] == 2027
].copy()

scenario_summary = forecast_2027.pivot(
    index="ticker",
    columns="scenario",
    values=[
        "revenue",
        "operating_income",
        "free_cash_flow"
    ]
)

scenario_summary = scenario_summary.round(2)

display(scenario_summary)


In [ ]:

# Cell 70 — Validate and export scenario forecasts

assert len(forecast_assumptions) == 9
assert len(scenario_forecast) == 18

assert not scenario_forecast.isna().any().any()

assert (
    scenario_forecast["revenue"] > 0
).all()

assert (
    scenario_forecast["free_cash_flow"] > 0
).all()

forecast_assumptions.to_csv(
    "defense_forecast_assumptions.csv",
    index=False
)

scenario_forecast.to_csv(
    "defense_scenario_forecast_2026_2027.csv",
    index=False
)

print("PASS: All 18 scenario forecast records validated.")
print("Saved defense_forecast_assumptions.csv")
print("Saved defense_scenario_forecast_2026_2027.csv")


## Excel financial model and validation

Generate and validate the workbook. Some steps require downloading, recalculating, and re-uploading Excel files in Colab.

In [ ]:

# Cell 71 — Prepare Excel workbook tools

!pip -q install openpyxl

from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.chart import BarChart, Reference

print("Excel workbook tools ready.")


In [ ]:

# Cell 72 — Create workbook with historical financial data

wb = Workbook()

ws_history = wb.active
ws_history.title = "Historical Data"

ws_assumptions = wb.create_sheet("Assumptions")
ws_forecast = wb.create_sheet("Forecast")
ws_summary = wb.create_sheet("Scenario Summary")

history_columns = [
    "ticker",
    "year",
    "revenue",
    "revenue_growth_pct",
    "operating_income",
    "operating_margin_pct",
    "free_cash_flow",
    "fcf_margin_pct"
]

ws_history.append(history_columns)

for row in forecast_history[history_columns].itertuples(
    index=False,
    name=None
):
    ws_history.append([
        None if pd.isna(value) else value
        for value in row
    ])

print("Historical records loaded:", ws_history.max_row - 1)


In [ ]:

# Cell 73 — Add scenario assumptions

assumption_headers = [
    "Ticker",
    "Scenario",
    "FY2025 Revenue ($B)",
    "Revenue Growth (%)",
    "Operating Margin (%)",
    "FCF Margin (%)"
]

ws_assumptions.append(assumption_headers)

assumption_columns = [
    "ticker",
    "scenario",
    "revenue",
    "revenue_growth_pct",
    "operating_margin_pct",
    "fcf_margin_pct"
]

for row in forecast_assumptions[
    assumption_columns
].itertuples(index=False, name=None):

    ws_assumptions.append(list(row))

print("Scenario assumption records:", ws_assumptions.max_row - 1)


In [ ]:

# Cell 74 — Create Excel forecast formulas

forecast_headers = [
    "Ticker",
    "Scenario",
    "Year",
    "Revenue ($B)",
    "Operating Income ($B)",
    "Free Cash Flow ($B)",
    "Revenue Growth (%)",
    "Operating Margin (%)",
    "FCF Margin (%)"
]

ws_forecast.append(forecast_headers)

for assumption_row in range(2, 11):

    # Two forecast years per scenario
    for year in [2026, 2027]:

        forecast_row = ws_forecast.max_row + 1

        ws_forecast.cell(
            forecast_row, 1,
            f"=Assumptions!A{assumption_row}"
        )

        ws_forecast.cell(
            forecast_row, 2,
            f"=Assumptions!B{assumption_row}"
        )

        ws_forecast.cell(forecast_row, 3, year)

        # Revenue forecast
        if year == 2026:
            revenue_formula = (
                f"=Assumptions!C{assumption_row}"
                f"*(1+Assumptions!D{assumption_row}/100)"
            )
        else:
            revenue_formula = (
                f"=D{forecast_row-1}"
                f"*(1+Assumptions!D{assumption_row}/100)"
            )

        ws_forecast.cell(
            forecast_row, 4,
            revenue_formula
        )

        # Operating income
        ws_forecast.cell(
            forecast_row, 5,
            f"=D{forecast_row}"
            f"*Assumptions!E{assumption_row}/100"
        )

        # Free cash flow
        ws_forecast.cell(
            forecast_row, 6,
            f"=D{forecast_row}"
            f"*Assumptions!F{assumption_row}/100"
        )

        # Link forecast assumptions
        for col, assumption_col in [
            (7, "D"),
            (8, "E"),
            (9, "F")
        ]:
            ws_forecast.cell(
                forecast_row,
                col,
                f"=Assumptions!{assumption_col}{assumption_row}"
            )

print("Formula-driven forecast rows:", ws_forecast.max_row - 1)


In [ ]:

# Cell 75 — Create scenario summary

summary_headers = [
    "Ticker",
    "Scenario",
    "FY2027 Revenue ($B)",
    "FY2027 Operating Income ($B)",
    "FY2027 Free Cash Flow ($B)"
]

ws_summary.append(summary_headers)

for row_idx in range(2, ws_forecast.max_row + 1):

    if ws_forecast.cell(row_idx, 3).value == 2027:

        summary_row = ws_summary.max_row + 1

        ws_summary.cell(
            summary_row, 1,
            f"=Forecast!A{row_idx}"
        )

        ws_summary.cell(
            summary_row, 2,
            f"=Forecast!B{row_idx}"
        )

        for summary_col, forecast_col in [
            (3, "D"),
            (4, "E"),
            (5, "F")
        ]:
            ws_summary.cell(
                summary_row,
                summary_col,
                f"=Forecast!{forecast_col}{row_idx}"
            )

print("FY2027 summary rows:", ws_summary.max_row - 1)


In [ ]:

# Cell 76 — Format and save workbook

navy = "17324D"
white = "FFFFFF"
input_blue = "E8F2FF"

for ws in wb.worksheets:

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:
        cell.fill = PatternFill(
            "solid",
            fgColor=navy
        )
        cell.font = Font(
            color=white,
            bold=True
        )
        cell.alignment = Alignment(
            horizontal="center"
        )

    for column_cells in ws.columns:

        letter = column_cells[0].column_letter

        max_length = max(
            len(str(cell.value or ""))
            if cell.data_type != "f" else 18
            for cell in column_cells
        )

        ws.column_dimensions[letter].width = min(
            max(max_length + 3, 15),
            34
        )

    for row in ws.iter_rows(
        min_row=2,
        min_col=3
    ):
        for cell in row:
            if cell.data_type == "f" or isinstance(
                cell.value, (int, float)
            ):
                if cell.column != 3 or ws.title != "Forecast":
                    cell.number_format = '#,##0.00;[Red](#,##0.00)'

# Highlight editable assumption inputs
for row in ws_assumptions.iter_rows(
    min_row=2,
    min_col=4,
    max_col=6
):
    for cell in row:
        cell.fill = PatternFill(
            "solid",
            fgColor=input_blue
        )

# Correct numeric formatting for historical year
for row in range(2, ws_history.max_row + 1):
    ws_history.cell(row, 2).number_format = "0"

# Correct numeric formatting for forecast year
for row in range(2, ws_forecast.max_row + 1):
    ws_forecast.cell(row, 3).number_format = "0"

# Request automatic formula recalculation in Excel
from openpyxl.workbook.properties import CalcProperties

wb.calculation = CalcProperties(
    calcMode="auto",
    fullCalcOnLoad=True,
    forceFullCalc=True
)

workbook_filename = "defense_financial_forecast_2026_2027.xlsx"

wb.save(workbook_filename)

print("PASS: Excel forecasting workbook created.")
print("Saved:", workbook_filename)
print("Worksheets:", wb.sheetnames)


In [ ]:

# Cell 77 — Verify Excel workbook

check_wb = load_workbook(
    "defense_financial_forecast_2026_2027.xlsx"
)

assert len(check_wb.sheetnames) == 4

assert check_wb["Historical Data"].max_row == 16
assert check_wb["Assumptions"].max_row == 10
assert check_wb["Forecast"].max_row == 19
assert check_wb["Scenario Summary"].max_row == 10

assert (
    check_wb["Forecast"]["D2"].data_type == "f"
)

assert (
    check_wb["Forecast"]["D3"].data_type == "f"
)

print("PASS: Excel workbook structure validated.")
print("PASS: Forecast formulas are present.")
print("PASS: All four worksheets are present.")


In [ ]:

# Cell 78 — Download the Excel forecasting workbook

from google.colab import files

files.download(
    "defense_financial_forecast_2026_2027.xlsx"
)


In [ ]:

# Cell 79 — Upload the workbook saved by Excel

from google.colab import files

uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))


In [ ]:

# Cell 80 — Validate Excel forecast against Python

from openpyxl import load_workbook
import numpy as np
import pandas as pd

uploaded_filename = next(iter(uploaded.keys()))

excel_wb = load_workbook(
    uploaded_filename,
    data_only=True
)

excel_ws = excel_wb["Forecast"]

excel_rows = list(
    excel_ws.iter_rows(
        min_row=2,
        values_only=True
    )
)

excel_forecast = pd.DataFrame(
    excel_rows,
    columns=[
        "ticker",
        "scenario",
        "year",
        "revenue",
        "operating_income",
        "free_cash_flow",
        "revenue_growth_pct",
        "operating_margin_pct",
        "fcf_margin_pct"
    ]
)

assert len(excel_forecast) == 18, (
    "Expected 18 Excel forecast records."
)

metrics = [
    "revenue",
    "operating_income",
    "free_cash_flow"
]

comparison = excel_forecast.merge(
    scenario_forecast,
    on=["ticker", "scenario", "year"],
    suffixes=("_excel", "_python"),
    validate="one_to_one"
)

assert len(comparison) == 18

for metric in metrics:

    excel_values = pd.to_numeric(
        comparison[f"{metric}_excel"],
        errors="coerce"
    )

    python_values = comparison[f"{metric}_python"]

    assert excel_values.notna().all(), (
        f"Excel has missing calculated values: {metric}"
    )

    assert np.allclose(
        excel_values,
        python_values,
        rtol=1e-9,
        atol=1e-8
    ), f"Excel and Python disagree on {metric}"

    print(f"PASS: Excel {metric} matches Python")

print("\nPASS: All 18 Excel forecasts match Python.")


In [ ]:

# Cell 81 — Diagnose Excel formula calculation

from openpyxl import load_workbook
import pandas as pd

uploaded_filename = next(iter(uploaded.keys()))

formula_wb = load_workbook(
    uploaded_filename,
    data_only=False
)

values_wb = load_workbook(
    uploaded_filename,
    data_only=True
)

formula_ws = formula_wb["Forecast"]
values_ws = values_wb["Forecast"]

print("Uploaded workbook:", uploaded_filename)

print("\nFormula inspection:")

for cell_address in ["A2", "B2", "D2", "E2", "F2"]:

    print(
        cell_address,
        "| Formula:",
        formula_ws[cell_address].value,
        "| Calculated value:",
        values_ws[cell_address].value
    )

print("\nCalculation status:")

missing_count = sum(
    values_ws.cell(row, 4).value is None
    for row in range(2, 20)
)

print("Forecast rows:", 18)
print("Rows missing calculated revenue:", missing_count)


In [ ]:

# Cell 82 — Install LibreOffice in Colab

!apt-get update -qq
!apt-get install -y -qq libreoffice-calc > /dev/null

print("Spreadsheet calculation engine installed.")


In [ ]:

# Cell 83 — Recalculate Excel formulas

import os
import subprocess

original_file = "defense_financial_forecast_2026_2027.xlsx"

output_directory = "/content/recalculated"
os.makedirs(output_directory, exist_ok=True)

result = subprocess.run(
    [
        "libreoffice",
        "-env:UserInstallation=file:///tmp/lo_defense_forecast",
        "--headless",
        "--convert-to",
        "xlsx",
        "--outdir",
        output_directory,
        os.path.abspath(original_file)
    ],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Output:", result.stdout)
print("Errors:", result.stderr)

recalculated_file = os.path.join(
    output_directory,
    original_file
)

assert os.path.exists(recalculated_file), (
    "Recalculated workbook was not created."
)

print("\nPASS: Recalculated workbook created.")


In [ ]:

# Cell 84 — Inspect recalculated Excel values

from openpyxl import load_workbook

calculated_wb = load_workbook(
    recalculated_file,
    data_only=True
)

calculated_ws = calculated_wb["Forecast"]

for cell_address in ["A2", "B2", "D2", "E2", "F2"]:
    print(
        cell_address,
        ":",
        calculated_ws[cell_address].value
    )

missing_count = sum(
    calculated_ws.cell(row, 4).value is None
    for row in range(2, 20)
)

print("\nRows missing calculated revenue:", missing_count)

assert missing_count == 0

print("PASS: All forecast revenue formulas calculated.")


In [ ]:

# Cell 85 — Compare calculated Excel results to Python

import pandas as pd
import numpy as np

excel_rows = list(
    calculated_ws.iter_rows(
        min_row=2,
        max_row=19,
        values_only=True
    )
)

excel_forecast = pd.DataFrame(
    excel_rows,
    columns=[
        "ticker",
        "scenario",
        "year",
        "revenue",
        "operating_income",
        "free_cash_flow",
        "revenue_growth_pct",
        "operating_margin_pct",
        "fcf_margin_pct"
    ]
)

assert len(excel_forecast) == 18

keys = ["ticker", "scenario", "year"]

assert not excel_forecast[keys].isna().any().any(), (
    "Missing company, scenario, or year."
)

assert not excel_forecast.duplicated(keys).any(), (
    "Duplicate Excel forecast records."
)

comparison = excel_forecast.merge(
    scenario_forecast,
    on=keys,
    suffixes=("_excel", "_python"),
    validate="one_to_one"
)

assert len(comparison) == 18

metrics = [
    "revenue",
    "operating_income",
    "free_cash_flow"
]

for metric in metrics:

    excel_values = pd.to_numeric(
        comparison[f"{metric}_excel"],
        errors="coerce"
    )

    python_values = comparison[f"{metric}_python"]

    assert excel_values.notna().all(), (
        f"Missing Excel calculations: {metric}"
    )

    assert np.allclose(
        excel_values,
        python_values,
        rtol=1e-9,
        atol=1e-8
    ), f"Mismatch: {metric}"

    print(f"PASS: Excel {metric} matches Python")

print("\nPASS: All 18 Excel forecasts match Python.")


In [ ]:

# Cell 86 — Save and download validated Excel workbook

import shutil
from google.colab import files

final_workbook = (
    "defense_financial_forecast_VALIDATED.xlsx"
)

shutil.copy2(
    recalculated_file,
    final_workbook
)

print("Saved:", final_workbook)

files.download(final_workbook)


## Free cash flow sensitivity analysis

Model FY2027 free cash flow across growth and margin assumptions and add sensitivity tables.

In [ ]:

# Cell 87 — Define sensitivity analysis ranges

growth_offsets = [-2, -1, 0, 1, 2]
fcf_margin_offsets = [-1, -0.5, 0, 0.5, 1]

sensitivity_rows = []

base_assumptions = forecast_assumptions[
    forecast_assumptions["scenario"] == "Base"
].copy()

for company in base_assumptions.itertuples(index=False):

    for growth_offset in growth_offsets:

        for margin_offset in fcf_margin_offsets:

            growth_rate = (
                company.revenue_growth_pct
                + growth_offset
            )

            fcf_margin = (
                company.fcf_margin_pct
                + margin_offset
            )

            revenue_2027 = (
                company.revenue
                * (1 + growth_rate / 100) ** 2
            )

            fcf_2027 = (
                revenue_2027
                * fcf_margin / 100
            )

            sensitivity_rows.append({
                "ticker": company.ticker,
                "revenue_growth_pct": growth_rate,
                "fcf_margin_pct": fcf_margin,
                "revenue_2027": revenue_2027,
                "fcf_2027": fcf_2027
            })

sensitivity_analysis = pd.DataFrame(
    sensitivity_rows
)

print(
    "Sensitivity analysis records:",
    len(sensitivity_analysis)
)

display(sensitivity_analysis.head())


In [ ]:

# Cell 88 — Lockheed Martin FY2027 FCF sensitivity

lmt_sensitivity = sensitivity_analysis[
    sensitivity_analysis["ticker"] == "LMT"
]

lmt_sensitivity_table = lmt_sensitivity.pivot(
    index="fcf_margin_pct",
    columns="revenue_growth_pct",
    values="fcf_2027"
)

lmt_sensitivity_table.index = (
    lmt_sensitivity_table.index.round(2)
)

lmt_sensitivity_table.columns = (
    lmt_sensitivity_table.columns.round(2)
)

print(
    "Lockheed Martin FY2027 Free Cash Flow "
    "Sensitivity ($ billions)"
)

display(lmt_sensitivity_table.round(2))


In [ ]:

# Cell 89 — Validate sensitivity analysis

assert len(sensitivity_analysis) == 75

for company in base_assumptions.itertuples(index=False):

    center_growth = company.revenue_growth_pct
    center_margin = company.fcf_margin_pct

    center_row = sensitivity_analysis[
        (sensitivity_analysis["ticker"] == company.ticker)
        & (
            abs(
                sensitivity_analysis["revenue_growth_pct"]
                - center_growth
            ) < 1e-9
        )
        & (
            abs(
                sensitivity_analysis["fcf_margin_pct"]
                - center_margin
            ) < 1e-9
        )
    ]

    assert len(center_row) == 1

    expected_fcf = scenario_forecast[
        (scenario_forecast["ticker"] == company.ticker)
        & (scenario_forecast["scenario"] == "Base")
        & (scenario_forecast["year"] == 2027)
    ]["free_cash_flow"].iloc[0]

    actual_fcf = center_row["fcf_2027"].iloc[0]

    assert abs(actual_fcf - expected_fcf) < 1e-8

    print(
        f"PASS: {company.ticker} sensitivity center "
        f"matches Base forecast (${actual_fcf:.2f}B)"
    )

print("\nPASS: All sensitivity tables validated.")


In [ ]:

# Cell 90 — Export sensitivity analysis

sensitivity_analysis.to_csv(
    "defense_fcf_sensitivity_2027.csv",
    index=False
)

print("Saved defense_fcf_sensitivity_2027.csv")
print("Records:", len(sensitivity_analysis))


In [ ]:

# Cell 91 — Open the validated workbook

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

workbook_path = "defense_financial_forecast_VALIDATED.xlsx"

wb = load_workbook(workbook_path)

for sheet_name in ["FCF Sensitivity", "Executive Summary"]:
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]

ws_sensitivity = wb.create_sheet("FCF Sensitivity")
ws_executive = wb.create_sheet("Executive Summary", 0)

print("Workbook sheets:", wb.sheetnames)


In [ ]:

# Cell 92 — Add dynamic sensitivity tables

growth_offsets = [-2, -1, 0, 1, 2]
margin_offsets = [-1, -0.5, 0, 0.5, 1]

company_names = {
    "GD": "General Dynamics",
    "LMT": "Lockheed Martin",
    "NOC": "Northrop Grumman"
}

for company_index, ticker in enumerate(["GD", "LMT", "NOC"]):

    start_row = 2 + company_index * 10

    assumption_row = next(
        row
        for row in range(2, 11)
        if ws_assumptions.cell(row, 1).value == ticker
        and ws_assumptions.cell(row, 2).value == "Base"
    )

    ws_sensitivity.cell(
        start_row, 1,
        f"{company_names[ticker]} — FY2027 FCF ($B)"
    )

    ws_sensitivity.cell(
        start_row + 1, 1,
        "FCF Margin / Revenue Growth"
    )

    # Growth assumptions across columns B:F
    for col_offset, growth_offset in enumerate(growth_offsets):

        col = col_offset + 2

        ws_sensitivity.cell(
            start_row + 1,
            col,
            f"=Assumptions!D{assumption_row}+({growth_offset})"
        )

    # FCF margin assumptions down rows
    for row_offset, margin_offset in enumerate(margin_offsets):

        row = start_row + 2 + row_offset

        ws_sensitivity.cell(
            row,
            1,
            f"=Assumptions!F{assumption_row}+({margin_offset})"
        )

        for col in range(2, 7):

            growth_cell = (
                f"{get_column_letter(col)}${start_row + 1}"
            )

            margin_cell = f"$A{row}"

            ws_sensitivity.cell(
                row,
                col,
                (
                    f"=Assumptions!$C${assumption_row}"
                    f"*(1+{growth_cell}/100)^2"
                    f"*{margin_cell}/100"
                )
            )

print("Created three dynamic FCF sensitivity tables.")


In [ ]:

# Cell 93 — Build executive summary

ws_executive["A1"] = "DEFENSE INDUSTRY FINANCIAL INTELLIGENCE"
ws_executive["A2"] = "FY2021–2025 Historical Analysis | FY2026–2027 Scenarios"

ws_executive["A4"] = "FY2027 BASE SCENARIO"

headers = [
    "Company",
    "Revenue ($B)",
    "Operating Income ($B)",
    "Free Cash Flow ($B)"
]

for col, header in enumerate(headers, start=1):
    ws_executive.cell(5, col, header)

for idx, ticker in enumerate(["GD", "LMT", "NOC"]):

    summary_row = 6 + idx

    forecast_row = next(
        row
        for row in range(2, ws_forecast.max_row + 1)
        if ws_forecast.cell(row, 3).value == 2027
        and ws_assumptions.cell(
            2 + (row - 2) // 2, 1
        ).value == ticker
        and ws_assumptions.cell(
            2 + (row - 2) // 2, 2
        ).value == "Base"
    )

    ws_executive.cell(
        summary_row, 1,
        company_names[ticker]
    )

    for col, forecast_col in [
        (2, "D"),
        (3, "E"),
        (4, "F")
    ]:
        ws_executive.cell(
            summary_row,
            col,
            f"=Forecast!{forecast_col}{forecast_row}"
        )

ws_executive["A10"] = "KEY HISTORICAL FINDINGS"

findings = [
    "General Dynamics: Highest FY2021–2025 revenue CAGR (8.11%).",
    "Lockheed Martin: Highest five-year cumulative free cash flow ($32.26B).",
    "Northrop Grumman: Highest FY2025 operating margin (10.75%).",
    "General Dynamics: Most stable operating margins across FY2021–2025."
]

for row, finding in enumerate(findings, start=11):
    ws_executive.cell(row, 1, finding)

ws_executive["A17"] = "FORECAST METHODOLOGY"
ws_executive["A18"] = (
    "Base assumptions use FY2023–2025 averages."
)
ws_executive["A19"] = (
    "Bear/Bull scenarios adjust growth by ±3 pp "
    "and operating/FCF margins by ±1 pp."
)
ws_executive["A20"] = (
    "Forecasts are illustrative scenarios, not company guidance."
)
ws_executive["A21"] = (
    "Historical SEC filing reconciliation remains pending."
)

print("Executive summary created.")


In [ ]:

# Cell 94 — Format and save expanded workbook

navy = "17324D"
light_blue = "E8F2FF"
white = "FFFFFF"

for ws in [ws_sensitivity, ws_executive]:

    ws.sheet_view.showGridLines = False

    ws.column_dimensions["A"].width = (
        70 if ws == ws_executive else 35
    )

    for col in range(2, 7):
        ws.column_dimensions[
            get_column_letter(col)
        ].width = 22

    ws.freeze_panes = "B2"

for row in [2, 12, 22]:
    for col in range(1, 7):
        cell = ws_sensitivity.cell(row, col)
        cell.fill = PatternFill(
            "solid",
            fgColor=navy
        )
        cell.font = Font(
            bold=True,
            color=white
        )

for row in [3, 13, 23]:
    for col in range(1, 7):
        cell = ws_sensitivity.cell(row, col)
        cell.fill = PatternFill(
            "solid",
            fgColor=light_blue
        )
        cell.font = Font(bold=True)

for row in range(1, 31):
    for col in range(2, 7):
        cell = ws_sensitivity.cell(row, col)
        if cell.data_type == "f":
            cell.number_format = "0.00"

for row in [1, 4, 10, 17]:
    cell = ws_executive.cell(row, 1)
    cell.font = Font(
        bold=True,
        color=navy,
        size=14 if row == 1 else 12
    )

for col in range(1, 5):
    cell = ws_executive.cell(5, col)
    cell.fill = PatternFill(
        "solid",
        fgColor=navy
    )
    cell.font = Font(
        bold=True,
        color=white
    )

for row in range(6, 9):
    for col in range(2, 5):
        ws_executive.cell(
            row, col
        ).number_format = '#,##0.00'

from openpyxl.workbook.properties import CalcProperties

wb.calculation = CalcProperties(
    calcMode="auto",
    fullCalcOnLoad=True,
    forceFullCalc=True
)

expanded_workbook = (
    "defense_financial_forecast_EXECUTIVE.xlsx"
)

wb.save(expanded_workbook)

print("Saved:", expanded_workbook)
print("Worksheet count:", len(wb.sheetnames))


In [ ]:

# Cell 95 — Validate workbook structure

check_wb = load_workbook(expanded_workbook)

assert len(check_wb.sheetnames) == 6

assert (
    check_wb["FCF Sensitivity"]["D6"].data_type == "f"
)

assert (
    check_wb["Executive Summary"]["B6"].data_type == "f"
)

print("PASS: Six worksheets present.")
print("PASS: Sensitivity formulas present.")
print("PASS: Executive summary formulas present.")


In [ ]:

# Cell 96 — Recalculate expanded Excel workbook

import os
import subprocess

output_directory = "/content/recalculated_final"
os.makedirs(output_directory, exist_ok=True)

result = subprocess.run(
    [
        "libreoffice",
        "-env:UserInstallation=file:///tmp/lo_defense_final",
        "--headless",
        "--convert-to",
        "xlsx",
        "--outdir",
        output_directory,
        os.path.abspath(expanded_workbook)
    ],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("Output:", result.stdout)
print("Errors:", result.stderr)

final_calculated_file = os.path.join(
    output_directory,
    expanded_workbook
)

assert result.returncode == 0
assert os.path.exists(final_calculated_file)

print("PASS: Expanded workbook recalculated.")


In [ ]:

# Cell 97 — Validate all Excel sensitivity calculations

from openpyxl import load_workbook
import numpy as np

calculated_wb = load_workbook(
    final_calculated_file,
    data_only=True
)

sensitivity_ws = calculated_wb["FCF Sensitivity"]

company_blocks = {
    "GD": 2,
    "LMT": 12,
    "NOC": 22
}

validated_cells = 0

for ticker, start_row in company_blocks.items():

    company_data = sensitivity_analysis[
        sensitivity_analysis["ticker"] == ticker
    ]

    for row_offset in range(5):

        excel_row = start_row + 2 + row_offset

        margin = sensitivity_ws.cell(
            excel_row, 1
        ).value

        for col_offset in range(5):

            excel_col = col_offset + 2

            growth = sensitivity_ws.cell(
                start_row + 1,
                excel_col
            ).value

            excel_fcf = sensitivity_ws.cell(
                excel_row,
                excel_col
            ).value

            matching_row = company_data[
                np.isclose(
                    company_data["revenue_growth_pct"],
                    growth,
                    atol=1e-8
                )
                &
                np.isclose(
                    company_data["fcf_margin_pct"],
                    margin,
                    atol=1e-8
                )
            ]

            assert len(matching_row) == 1, (
                f"Missing Python match for {ticker}"
            )

            expected_fcf = matching_row[
                "fcf_2027"
            ].iloc[0]

            assert np.isclose(
                excel_fcf,
                expected_fcf,
                rtol=1e-9,
                atol=1e-8
            ), (
                f"Sensitivity mismatch: "
                f"{ticker}, row {excel_row}, "
                f"column {excel_col}"
            )

            validated_cells += 1

    print(f"PASS: {ticker} sensitivity table matches Python.")

assert validated_cells == 75

print(
    "\nPASS: All 75 Excel sensitivity "
    "calculations match Python."
)


In [ ]:

# Cell 98 — Validate executive summary forecasts

executive_ws = calculated_wb["Executive Summary"]

company_rows = {
    "GD": 6,
    "LMT": 7,
    "NOC": 8
}

metric_columns = {
    "revenue": 2,
    "operating_income": 3,
    "free_cash_flow": 4
}

for ticker, excel_row in company_rows.items():

    expected = scenario_forecast[
        (scenario_forecast["ticker"] == ticker)
        & (scenario_forecast["scenario"] == "Base")
        & (scenario_forecast["year"] == 2027)
    ].iloc[0]

    for metric, excel_col in metric_columns.items():

        excel_value = executive_ws.cell(
            excel_row,
            excel_col
        ).value

        python_value = expected[metric]

        assert np.isclose(
            excel_value,
            python_value,
            rtol=1e-9,
            atol=1e-8
        ), f"Mismatch: {ticker}, {metric}"

    print(f"PASS: {ticker} executive summary matches Python.")

print("\nPASS: All executive summary forecasts validated.")


In [ ]:

# Cell 99 — Download final validated Excel workbook

import shutil
from google.colab import files

final_filename = (
    "defense_financial_intelligence_forecast_FINAL.xlsx"
)

shutil.copy2(
    final_calculated_file,
    final_filename
)

print("Final Excel workbook:", final_filename)
print("Size (KB):", round(
    os.path.getsize(final_filename) / 1024,
    1
))

files.download(final_filename)


## Power BI data exports

Export structured CSV datasets for the Power BI report.

In [ ]:

# Cell 100 — Create Power BI data package

import os
import shutil

powerbi_folder = "defense_powerbi_data"

os.makedirs(powerbi_folder, exist_ok=True)

powerbi_files = [
    "defense_peer_financial_ratios.csv",
    "defense_peer_scorecard.csv",
    "defense_sql_peer_rankings.csv",
    "defense_scenario_forecast_2026_2027.csv",
    "defense_forecast_assumptions.csv",
    "defense_fcf_sensitivity_2027.csv"
]

for filename in powerbi_files:

    assert os.path.exists(filename), (
        f"Missing required file: {filename}"
    )

    shutil.copy2(
        filename,
        os.path.join(powerbi_folder, filename)
    )

    print("Added:", filename)

print("\nPASS: Power BI data package prepared.")


In [ ]:

# Cell 101 — Download Power BI data package

from google.colab import files

zip_path = shutil.make_archive(
    "defense_powerbi_data",
    "zip",
    powerbi_folder
)

print("Created:", zip_path)

files.download(zip_path)
